# Día 4 — Modelo de Regresión Lineal Múltiple

**Notebook:** `03_regresion.ipynb`  
**Responsable:** Patricio, Gerardo
**Materia:** Tópicos Selectos de Ciencias de la Ingeniería 3 (UANL, FIME)  

En este notebook avanzamos hacia la fase de modelado predictivo, tomando como base el análisis de correlación del Día 3. Ya sabemos qué variables están asociadas linealmente con los costos médicos (`charges`); ahora construiremos un modelo de **Regresión Lineal Múltiple** para cuantificar el impacto simultáneo de todas las variables independientes sobre la variable dependiente y evaluar la capacidad predictiva global del sistema.

Trabajaremos con el dataset ya codificado en formato numérico para incluir las variables categóricas (`sex`, `smoker`, `region`) mediante variables *dummy*.

## 0. Carga de los datos

Se carga el dataset previamente codificado en formato nuérico proveniente de `df_numerico` convertido a un archivo *.csv* para su correcta importación

In [22]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

dfn = pd.read_csv("..\data\insurance_processed_d3.csv") 
dfn.head()

<>:5: SyntaxWarning: invalid escape sequence '\d'
<>:5: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Pato\AppData\Local\Temp\ipykernel_15200\1739234646.py:5: SyntaxWarning: invalid escape sequence '\d'
  dfn = pd.read_csv("..\data\insurance_processed_d3.csv")


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,1,27.9000,0,1,16884.9240,0,0,1
1,18,0,33.7700,1,0,1725.5523,0,1,0
2,28,0,33.0000,3,0,4449.4620,0,1,0
3,33,0,22.7050,0,0,21984.4706,1,0,0
4,32,0,28.8800,0,0,3866.8552,1,0,0


## 1. División de Variables (Independientes vs. Dependiente)

Para entrenar el modelo, separamos las características predictoras en la matriz $X$ y la variable objetivo en el vector $Y$. 

* **Variable Dependiente ($Y$):** `charges` (Costos médicos de los pacientes).
* **Variables Independientes ($X$):** `age`, `sex`, `bmi`, `children`, `smoker` y las columnas *dummy* de región (`region_northwest`, `region_southeast`, `region_southwest`). 

    * *Nota:* Para evitar la trampa de la multicolinealidad (trampa de variables dummy), excluimos una de las regiones (por ejemplo, `region_northeast`), que actuará como nuestra categoría de referencia basal.

In [15]:
# Definir variables independientes (X) y dependiente (Y)
X = dfn[['age', 'sex', 'bmi', 'children', 'smoker', 'region_northwest', 'region_southeast', 'region_southwest']]
Y = dfn['charges']

# Statsmodels requiere añadir una constante (el intercepto beta_0)
X = sm.add_constant(X)

print("Dimensiones de X:", X.shape)
print("Dimensiones de Y:", Y.shape)

Dimensiones de X: (1337, 9)
Dimensiones de Y: (1337,)


## 2. Entrenamiento del Modelo de Regresión Lineal Múltiple

Utilizaremos el método de Mínimos Cuadrados Ordinarios (OLS - *Ordinary Least Squares*) a través de la librería `statsmodels`. Al ejecutar el método `.summary()`, se desplegará de forma cruda la tabla estadística.

In [21]:
# Ajustar el modelo OLS
modelo = sm.OLS(Y, X).fit()

# Desplegar el resultado bruto
print(modelo.summary2(float_format="%.4f"))

                       Results: Ordinary least squares
Model:                   OLS                 Adj. R-squared:        0.749     
Dependent Variable:      charges             AIC:                   27094.2154
Date:                    2026-07-08 03:09    BIC:                   27140.9991
No. Observations:        1337                Log-Likelihood:        -13538.   
Df Model:                8                   F-statistic:           500.0     
Df Residuals:            1328                Prob (F-statistic):    0.00      
R-squared:               0.751               Scale:                 3.6776e+07
------------------------------------------------------------------------------
                    Coef.     Std.Err.    t     P>|t|     [0.025      0.975]  
------------------------------------------------------------------------------
const            -12066.0390 1000.1973 -12.0637 0.0000 -14028.1780 -10103.9000
age                 256.7646   11.9122  21.5547 0.0000    233.3958    280.13

## 3. Análisis e Interpretación de Resultados

### R-cuadrado ($R^2$)
El coeficiente de determinación del modelo es de **0.751**. Esto significa que el **75.1%** de la variabilidad o cambios en los costos médicos (`charges`) se puede explicar y predecir de forma conjunta a través de las variables independientes incluidas en el modelo. El porcentaje restante se debe a factores aleatorios o variables no medidas.

### Coeficientes e Interpretación
Analizando los coeficientes ($\beta$) obtenidos para cada variable:
* **const (Intercepto):** Con un costo de \$`-256.7646` USD que representa el costo médico base estimado para un paciente cuando todas las variables predictoras son cero.
* **age:** Por cada año más de edad que tenga el paciente, los cargos médicos aumentan en promedio en \$`256.7646` USD, manteniendo todo lo demás constante.
* **smoker:** Si el paciente es fumador, sus costos médicos se incrementan drásticamente en \$`23847.3288` USD en comparación con un no fumador.
* **bmi:** Por cada unidad que aumente el Índice de Masa Corporal, los costos suben \$`339.2504` USD.
* **children:**  Por cada hijo o dependiente adicional que tenga registrado el asegurado, los cargos médicos estimados aumentan en promedio \$`474.8205` USD, manteniendo las demás variables constantes.
* **region_northwest:** Vivir en la región noroeste disminuye los cargos médicos en promedio \$`-349.2265` USD en comparación con la región basal de referencia (northeast), manteniendo todo lo demás constante[cite: 1].
* **region_southeast:** Vivir en la región sureste tiene un impacto de \$`-1035.2656` USD en los costos médicos en comparación con la región basal (northeast)[cite: 1].
* **region_southwest:** Los asegurados que residen en la región suroeste presentan una diferencia promedio de \$`-960.0814` USD en sus cargos médicos en comparación con la región de referencia (northeast)[cite: 1].

### P-values y Significancia Estadística ($\alpha = 0.05$)
Evaluando la columna `P>|t|` de la tabla:
* **Variables Estadísticamente Significativas ($p < 0.05$):** `age`, `bmi`, `smoker` tienen un p-value de `0.000`, lo que demuestra que tienen un impacto real y sistemático en los costos médicos.
* **Variables No Significativas ($p \ge 0.05$):** El p-value de las regiones como la variable `sex` si son mayores a 0.05, esto significa que, bajo este modelo lineal, esas variables no tienen un impacto estadísticamente relevante para predecir el costo médico.

## 4. Conclusión

El modelo de regresión lineal múltiple obtenido presenta un desempeño **sólido y robusto]** al dar como resultado un R² cercano a `0.75`. 

Con base estricta en los coeficientes calculados, la variable que predice de forma más masiva e importante el incremento de los costos médicos es **ser fumador (`smoker`)**. Esta característica desplaza los costos de manera radical en comparación con el impacto gradual que tienen la edad (`age`) o el índice de masa corporal (`bmi`), confirmando de manera analítica las sospechas iniciales obtenidas en la fase exploratoria y de correlación.

## Referencias

IBM. *Cognos Analytics*. (2025). https://www.ibm.com/docs/es/cognos-analytics/11.2.x?topic=tests-multiple-linear-regression

GeeksforGeeks. (2026, 2 mayo). *Multiple Linear Regression using Python ML. GeeksforGeeks*. https://www-geeksforgeeks-org.translate.goog/machine-learning/ml-multiple-linear-regression-using-python/?_x_tr_sl=en&_x_tr_tl=es&_x_tr_hl=es&_x_tr_pto=tc&_x_tr_hist=true

Minitab Blog Editor. (2024). *Análisis de regresión: ¿Cómo puedo interpretar el R-cuadrado y evaluar la bondad de ajuste?* https://blog.minitab.com/es/blog/analisis-de-regresion-como-puedo-interpretar-el-r-cuadrado-y-evaluar-la-bondad-de-ajuste